# Module 1 Note —— 量化分析入门

> 要理解量化，先要明白它的基础是什么，它优秀在哪里。

> 在金融领域，有三种基础的分析方式，分别为：**基本面分析**、**技术面分析**、**量化分析**。

> 我们先不要像经济学假设那样加上众多的限制，先只考虑 common sense。

## 第一部分 · 基础统计量

### 一、return（回报）怎么算

假设 $S$ 为股票价格，$R$ 为回报（收益），那么按常识可得出：

$$R_i = \frac{S_{i+1} - S_i}{S_i}$$

这是最简单的回报计算公式。

### 二、mean（均值）怎么算

就是正常的数学公式（注意：下面的统计量均作用于**回报** $R_i$，而不是价格 $S_i$）：

$$\mu = \frac{\sum_{i=1}^{n} R_i}{n}$$

### 三、std（样本标准差）怎么算

就是正常的数学公式（注：此处 $n-1$ 为贝塞尔校正，减小偏差）：

$$\sigma = \sqrt{\frac{\sum_{i=1}^{n} (R_i - \mu)^2}{n-1}}$$

### 四、normalize（归一化）处理

最终要使得 mean $= 0$，std $= 1$，我们需要构建新变量便于比较和计算：

$$\Phi = \frac{R_i - \mu}{\sigma}$$

### 五、distribution（分布）构建

就用标准正态分布公式：

$$f(\Phi) = \frac{1}{\sqrt{2\pi}} e^{-\frac{\Phi^2}{2}}$$

到这里，我们说 $\Phi$ 是服从标准正态分布的一个随机变量。

### 六、回报模型的整理与统一

整理上面第四个式子，得：

$$R_i = \frac{S_{i+1} - S_i}{S_i} = \mu + \sigma \cdot Z, \quad Z \sim N(0,1)$$

其中 $Z$ 为标准正态变量，其实就是 $\Phi$。这里的 $\Phi$（即 $Z$）是模型的**随机噪声源**，它与任何其他资产、任何其他影响因素都完全独立、互不相关，这也是本模型成立的前提假设。

> 比如大宗商品，因为与其相关的东西太多，不易分离，就不适合这种数学建模。

## 第二部分 · 时间尺度与连续化

### 七、时间对模型的影响

前面的模型显而易见是以天计的，我们把时间尺度缩小，先看一天：

$$\text{mean} = \mu \cdot \delta t$$

其中 $\mu$ 是年收益率（单位时间漂移率），$\delta t$ 就是一个极小的时间段（这里指一天），$\text{mean}$ 是平均日收益率。

在 $\delta t$ 足够小时，用"正常差值"计算回报与用"年化收益率差分"计算回报在**一阶近似**意义下相等（这是引入时间变量的必要一步联系）：

$$\bar{R} = \mu \cdot \delta t \quad \Longrightarrow \quad \frac{S_{i+1} - S_i}{S_i} = \mu \cdot \delta t$$

> 注：上式并非严格等式，而是 $\ln(1+x) \approx x$（即 $e^x \approx 1+x$）的一阶近似；$\delta t$ 越小误差越小（误差是 $\delta t$ 的二阶小量）。

于是：

$$S_{i+1} = S_i \cdot (1 + \mu \cdot \delta t)$$

令特定时间段 $t = M \cdot \delta t$，则由递归思想得：

$$S_M = S_0 \cdot (1 + \mu \cdot \delta t)^M$$

巧妙的复利思想再次体现！！

时间段如果足够小，就是极限嘛。两边求极限得（凑重要极限）：

$$S(t) = \lim_{M \to \infty} S_0 \left( 1 + \frac{\mu t}{M} \right)^M = S_0 \cdot e^{\mu t}$$

其实也可以表示成对数形式，对于 $\ln(1 + \mu \cdot \delta t)$，由泰勒展开近似为 $\mu \cdot \delta t$，细节不再赘述。

#### 时间膨胀系数 $\alpha$

前面其实有个隐形假设：收益与时间成正比，即完整的写法 $\delta t^{\alpha}$ 中 $\alpha = 1$。若 $\alpha \neq 1$ 呢？（$\alpha$ 是时间膨胀系数）

若单步收益率为 $\mu \delta t^{\alpha}$，则经过 $M = t / \delta t$ 步之后：

$$S(t) = S_0 \left( 1 + \mu \delta t^{\alpha} \right)^{t / \delta t} \approx S_0 \, e^{\mu t \, \delta t^{\alpha - 1}}$$

即指数上会在 $\mu t$ 后多乘一个 $\delta t^{\alpha - 1}$，那就可以分类讨论了：

- 如果 $\alpha < 1$，$\delta t^{\alpha - 1}$ 是**负指数**幂。$\delta t$ 是个足够小的数，$\delta t \to 0$ 时指数 $\mu t \, \delta t^{\alpha - 1} \to +\infty$，对应的 $e^x \to +\infty$，现实意义就是**收益无穷大**（模型发散、失效）。

- 如果 $\alpha > 1$，$\delta t^{\alpha - 1}$ 是**正指数**幂。$\delta t \to 0$ 时指数 $\mu t \, \delta t^{\alpha - 1} \to 0$，对应的 $e^x \to 1$，现实意义就是**收益无穷小，买与不买一个样**。

只有 $\alpha = 1$ 才给出有限且非平凡的结果 $S_0 e^{\mu t}$——这就是标准模型假设收益与时间成正比的原因。

### 八、方差（标准差）与时间的联系

衡量收益，一个重要的东西就是方差（标准差的平方），它该如何与时间建立联系？

假设每天收益率的标准差均为 $0.01$（即每天方差为 $10^{-4}$），连续 5 天的标准差会是多少？

我们知道，对于独立事件，方差具有线性可加性：5 天总方差 $= 5 \times 0.01^2$，则 5 天标准差为：

$$\sqrt{5 \times 0.01^2} = \sqrt{5} \cdot 0.01$$

这正是时间膨胀系数 $\alpha = 1/2$ 的情况：**方差随时间线性增长，标准差随时间以平方根速度增长**（时间系数不可能为负）。即：

$$\text{Var}(\delta t) = \sigma^2 \cdot \delta t, \qquad \text{std}(\delta t) = \sigma \cdot \delta t^{1/2}$$

### 九、资产价格的离散模型

综上所有，分别代入变量得：

$$R_i = \mu \cdot \delta t + \sigma \cdot \Phi \cdot \delta t^{1/2} = \frac{S_{i+1} - S_i}{S_i}$$

这就是**资产价格的离散模型**。

整理一下：

$$S_{i+1} = S_i \cdot (1 + \mu \cdot \delta t) + \sigma \cdot S_i \cdot \Phi \cdot \delta t^{1/2}$$

这不就是说，下一步资产的价值就等于趋势升值加上波动升值嘛。

**这个，就是大名鼎鼎的蒙特卡洛模拟。**（每次模拟抽取一个 $\Phi \sim N(0,1)$，代入上式迭代，即可生成一条价格路径）

> 注：上式是"加法噪声"的简化离散化。更严谨的对数离散化是 $S_{i+1} = S_i \exp\left( \mu \delta t + \sigma \Phi \sqrt{\delta t} \right)$，两者在 $\delta t$ 很小、$\sigma \sqrt{\delta t}$ 不大时一阶近似相等。

### 十、从离散模型到连续模型（随机微分方程）

好，既然有了离散模型，我们就要开始把它变成连续模型了。

我们注意到，$\mu$ 和 $\sigma$ 乘上的 $\delta t$ 的指数并不相同：漂移项是 $\delta t$ 的**一阶**小量，波动项是 $\delta t$ 的**平方根阶**小量。当 $\delta t \to 0$ 时，$\delta t^{1/2} \gg \delta t$，所以**极短时间内波动项占主导**；而随着时间拉长，漂移项 $\mu t$ 线性累积，最终**长期趋势占主导**——这就是"短期波动优先、长期趋势优先"的底层数学原理。

$$S_{i+1} - S_i = \mu \cdot S_i \cdot \delta t + \sigma \cdot S_i \cdot \Phi \cdot \delta t^{1/2}$$

我们把时间尺度看的无穷小，令 $\delta t \to 0$：

$$dS = S_{i+1} - S_i, \qquad dX = \Phi \cdot \sqrt{dt}$$

右边就把 $S_i$ 直接写作 $S$，就变成了：

$$dS = \mu S \, dt + \sigma S \, dX$$

**这个，就是 SDE（随机微分方程）。**

其中 $dX$ 就是随机微分（一个随机变量！不是黎曼积分中确定的某值）。注意它的分布：

$$E[dX] = 0, \quad E[(dX)^2] = dt, \quad \text{Var}[dX] = dt$$

即 $dX \sim N(0, dt)$——它**不服从标准正态分布**（$dX / \sqrt{dt}$ 才是 $N(0,1)$）。更准确地说，$dX = X_{t+dt} - X_t$ 是布朗运动 $X_t$ 的**增量**：均值为 0，即长期来看对资产的变动趋势无影响；方差为 $dt$，即在极小的时间会对资产造成影响，造成的影响也是完全随机的。

> 随机微积分弥补了以前金融建模的动态部分的缺憾，成功将模型更细致地映射到了现实世界中。

> asset（资产）是一种统称，对很多商品都适用。其并非适用于所有，比如利率，虽然可以用，但就是不匹配。

> 这只是定价用的，不适用于风控等等。

> 所有不严谨的东西都将后续补充。

## 第三部分 · 二叉树模型与期权定价

> **二叉树模型（随机游走的极端简化模型）——期权定价**

> 能够体现金融的两大核心思想：**对冲**和**无套利**，同时能够给出期权定价。

> 能通过它简化复杂的金融思想。

### 十一、搭建一个简单的框架

我们先搭建一个非常简单的框架作为后续分析的基础：

- 一支股票 a stock
- 一个看涨期权 a call option tomorrow（欧式期权，到期才可行权）
- 涨跌幅已知
- 利率 $r = 0$（未来的折现就不用考虑利率了）

股价 $S = 100$：

- 概率 $p = 0.6$，上涨到 $101$
- 概率 $p = 0.4$，下降到 $99$
- 时间为 1 天

然后我们手里有一个看涨期权：

$$V = \max\{S_T - K, 0\}$$

行权价 $K = 100$。如果上涨，作为买方（long 方），肯定行权，挣了 1 元；下跌肯定不行权，挣了 0 元。（无期权费）

那我们有这么多信息，怎么算**现在**期权值多少钱呢？

一般我们会认为：期权价格是期望值：

$$V = \frac{1 \times 0.6 + 0 \times 0.4}{1 + r} = 0.6$$

（$1 + r$ 是折价回现在）

注意：这是用**真实概率**算期望的朴素定价，它**忽略了风险溢价**。现实定价必然包含风险溢价，即：

$$V = \frac{1 \times 0.6 + 0 \times 0.4}{1 + r + \text{risk}}$$

这个 risk 每个人认为都不同。但至少，分母变大，数变小，所以至少一定 $< 0.6$。

要处理 risk，两种方法：**对冲**和**风险中性**。

### 十二、Δ对冲：构建投资组合

非常简单，来一个期权和 $\frac{1}{2}$ 份股票的组合（原因后续说）。

那么 0 时刻成本（投资组合价值）是 $V - \frac{1}{2} \times 100$（买期权花了 $V$ 元，又卖出 $\frac{1}{2}$ 份股票获得 50 元，也就是做空）。

由上述可知，资产组合的价值为：

$$S^{+} = 1 - \frac{1}{2} \times 101 = -49.5$$

$$S^{-} = 0 - \frac{1}{2} \times 99 = -49.5$$

即股票价格不论上涨或下跌，最终价值一样，上涨或下跌的风险就被完全抵消了。

现在我知道明天我的资产总价值为 $-49.5$，那我怎么知道今天我的资产总价值具体是多少？

注意：无风险的场景资产现值与终值相等（前提是 $r = 0$），所以我们可以知道今天总价值肯定也是 $-49.5$。

趣事：就算你把钱放在枕头底下，明天也是这些钱，所以我们投资组合想实现无套利就必须终值等于现值。要不然就有了：承担极少的风险来套利，获得无风险的回报。

由于市场在不断的调节，有套利空间人们就会蜂拥而至，导致利差被抹平，最终依旧是无套利。所以无风险价格就一定是最终价格。

综上算得，$V - 50 = -49.5$，即 $V = 0.5$ 才是最好的。

意外发现：真实概率 $p$ 在计算过程中完全没有被用到，得出反直觉结论：**期权定价与股票的真实涨跌概率无关**，只取决于涨跌幅度（波动率）。（因为涨跌的风险已被对冲抵消）

我们其实应该关注涨跌幅度（波动率），因为波动越大，期权价格越高。

#### 1. 有套利空间，怎么套利？我们算出的价格为什么一定是最终价格？

假设市场上实际期权价格 $< 0.5$，我们算出理论定价 $0.5$，证明现在在市场上买期权非常划算（比预计少花钱了），继续上述过程获得利润。

假设市场上实际期权价格 $> 0.5$，我们算出理论定价 $0.5$，证明现在在市场上卖期权非常划算（能卖更多的钱），继续上述过程获得利润。

**举例一**：$V = 0.6$，$S = 100$（市场价高于理论价 → 卖出期权并买入股票）

| 时刻 | 操作 | 现金流 |
| --- | --- | --- |
| $t = 0$ | 卖期权（空头） | $+0.6$ |
| $t = 0$ | 买股票（多头） | $-\frac{1}{2} \times 100$ |
| $t = 0$ | 期初净现金流 | $-49.4$ |
| 涨（$S = 101$） | short 方赔付 | $-1$ |
| 涨（$S = 101$） | 卖出股票 | $+\frac{1}{2} \times 101$ |
| 涨（$S = 101$） | 期末净现金流 | $+49.5$ |
| 跌（$S = 99$） | short 方赔付 | $+0$ |
| 跌（$S = 99$） | 卖出股票 | $+\frac{1}{2} \times 99$ |
| 跌（$S = 99$） | 期末净现金流 | $+49.5$ |

综上无论如何都是赚到了 $49.5 - 49.4 = 0.1$ 元。

**举例二**：$V = 0.4$，$S = 100$（市场价低于理论价 → 买入期权并卖空股票）

| 时刻 | 操作 | 现金流 |
| --- | --- | --- |
| $t = 0$ | 买期权（多头） | $-0.4$ |
| $t = 0$ | 卖股票（空头） | $+\frac{1}{2} \times 100$ |
| $t = 0$ | 期初净现金流 | $+49.6$ |
| 涨（$S = 101$） | long 方行权 | $+1$ |
| 涨（$S = 101$） | 买回股票 | $-\frac{1}{2} \times 101$ |
| 涨（$S = 101$） | 期末净现金流 | $-49.5$ |
| 跌（$S = 99$） | long 方不行权 | $+0$ |
| 跌（$S = 99$） | 买回股票 | $-\frac{1}{2} \times 99$ |
| 跌（$S = 99$） | 期末净现金流 | $-49.5$ |

综上无论如何都是赚到了 $49.6 - 49.5 = 0.1$ 元。

来来回回，策略的本质都一样：**用期权与股票构造一个与股价涨跌无关的"锁定"组合**，期末组合价值确定，期初即可保证赚到确定的价差。

> 要知道，现实生活中肯定会有更多因素，比如滑点，情绪化交易等。

#### 2. 为什么是配置 $\frac{1}{2}$ 的股票？不能是其他数字吗？

设 $\Delta$ 为对冲比率，表示每持有一份期权，需卖出标的资产（一般指股票）的份数，反映了期权价格对于股票价格变化的敏感度，其离散形式即：

$$\Delta = \frac{\partial V}{\partial S}$$

那么根据上面的框架，我们得出：

- 初始成本：$V - \Delta \times 100$
- 股价上涨：$1 - \Delta \times 101$
- 股价下跌：$0 - \Delta \times 99$

要保证股价无论上涨还是下跌，最终收益不变，则上涨和下跌价值必然相等：

$$1 - 101\Delta = 0 - 99\Delta \quad \Longrightarrow \quad \Delta = 0.5$$

推广一下：

$$\Delta = \frac{\text{期权收益范围}}{\text{股价变化范围}} = \frac{V^{+} - V^{-}}{S^{+} - S^{-}}$$

注意：对冲是宽泛的，但是 Δ对冲一般指期权和股票的组合。

#### 3. 无风险回报率（利率）$r \neq 0$，怎么办？

就只需要把未来的价值折现到现在就好了，但需要知道一年有 252 个交易日，所以 $r = \frac{0.1}{252}$（此处 $0.1$ 是年利率）。

#### 4. 现在期权定价基本结束了，我们可以反思一下，股票价格真的合理吗？

之前用真实概率算得未来股价期望是 $0.6 \times 101 + 0.4 \times 99 = 100.2$，现在定价 $100$，相当于去掉了 $0.2$ 的风险补偿，其实是非常合理的。

**完全市场**：非常理想的市场。没有税收、资产可无限分割等乌托邦式市场。

在完全市场里，我们说：

$$V - \Delta \cdot S = \text{RFA (Risk Free Asset)}$$

就是资产组合的价值等于无风险资产的价值。

所以期权收益也就是投资收益与无风险资产收益的总和，或者说期权是完全由 payoff 决定的；难以计算期权价格时，也可以计算标的资产价值来估算期权价值。(1)

### 十三、真实世界与风险中性世界

| | 真实世界 | 风险中性世界 |
| --- | --- | --- |
| 风险态度 | 人对风险很敏感，风险越高回报越大 | 人承认风险但不关心风险 |
| 风险溢价 | 人没有办法统一风险溢价，所以没有办法用期望定价 | 人不承认风险溢价 |
| 定价方法 | 人可以通过风险对冲来定价 | 人可以通过无风险回报率来直接定价 |

怎样从真实世界转到风险中性世界？

还是刚才的框架，但风险中性世界中概率不一样了（注意这里 $r = 0$，故不必折现）：

$$E[S] = p' \times 101 + (1 - p') \times 99 = 100, \quad 得 \quad p' = 0.5$$

则风险中性概率 $p' = 0.5$，这样就可以保证风险中性世界中 $t = 0$ 时刻股票定价与真实世界定价完全一样。

期权价格怎么算？这时候可以用期望了，因为风险中性世界以无风险利率折现：

$$V = \frac{0.5 \times 1 + 0.5 \times 0}{1 + r} = 0.5$$

与之前在真实世界的定价完全相同！就算有 $r$，也好算，折现就行了。

好的，这样的话就两种方法都讲完了，显然他们殊途同归，那怎么联合起来呢？

设 $u$ 为上涨因子，则 $S^{+} = u \cdot S$，$u > 1$；

设 $v$ 为下跌因子，则 $S^{-} = v \cdot S$，$0 < v < 1$；

真实世界概率为 $p$，时间段（时间步长）为 $\delta t$。

### 十四、期权定价通用公式：二叉树模型

#### 一、将 Δ对冲与风险中性世界结合起来

对于某一投资组合：

$$\Pi = V - \Delta \cdot S$$

由上述假设：

$$\Delta = \frac{V^{+} - V^{-}}{(u - v) \cdot S}$$

再把 $\Delta$ 反代入原来的上涨或下跌的投资组合公式，得：

$$\Pi^{+} = V^{+} - \frac{u \cdot (V^{+} - V^{-})}{u - v}$$

$$\Pi^{-} = V^{-} - \frac{v \cdot (V^{+} - V^{-})}{u - v}$$

如果有无风险回报率 $r$ 呢——提出**折现因子**概念。

首先，由上面的时间对资产收益率的影响 $S(t) = S_0 \cdot e^{\mu t}$ 得：

$$\Pi_{\text{future}} = \Pi_{\text{now}} \cdot e^{\mu t}$$

那么对于现在来说：

$$\Pi_{\text{now}} = \frac{1}{e^{\mu t}} \cdot \Pi_{\text{future}}$$

由于组合无风险，收益率 $\mu$ 就是无风险利率（收益率）$r$；时间段 $t$ 就是 $\delta t$。

根据泰勒展开，在 $\delta t$ 足够小时，$e^{\mu t} \approx 1 + r \cdot \delta t$，所以折现因子就是：

$$\frac{1}{1 + r \cdot \delta t}$$

在无风险情况下：

$$V - \frac{V^{+} - V^{-}}{u - v} = \frac{1}{1 + r \cdot \delta t} \cdot \left( V^{+} - \frac{u \cdot (V^{+} - V^{-})}{u - v} \right)$$

注：代 $\Pi^{+}$ 和 $\Pi^{-}$ 都行，反正都相等。

故：

$$V = \frac{V^{+} - V^{-}}{u - v} + \frac{1}{1 + r \cdot \delta t} \cdot \frac{u \cdot V^{-} - v \cdot V^{+}}{u - v}$$

再整理一下：

$$V = \frac{1}{1 + r \cdot \delta t} \cdot \left[ p' \cdot V^{+} + (1 - p') \cdot V^{-} \right], \quad p' = \frac{1 + r \cdot \delta t - v}{u - v}$$

#### 二、结合 SDE：确定 $u$、$v$、$p$

我们知道在 SDE 里，重要的参数是 $\mu$ 和 $\sigma$；而在二叉树模型里，参数是 $u$、$v$、$p$（真实概率）。我们必须在它们之间建立关系。

根据 SDE $dS = \mu S \, dt + \sigma S \, dX$ 得：在这样足够小的时间段 $\delta t$ 中，均值的变化量其实是 $\mu S \delta t$。

在二叉树模型中，我们要计算这样的变化量，就是 $\delta t$ 时间后股价的期望减去最初的股价。列式为：

$$\left( p \cdot u \cdot S + (1 - p) \cdot v \cdot S \right) - S = \mu S \delta t$$

得不出来什么，我们还需要其它式子。

因为（$dS$ 的方差来自波动项：漂移项 $\mu S dt$ 是确定性的，方差为 0）：

$$\text{Var}(dS) = \text{Var}(\mu S dt) + \text{Var}(\sigma S dX) = 0 + \sigma^2 S^2 \text{Var}(dX) = \sigma^2 S^2 dt$$

相应的，二叉树模型的方差为：

$$\text{Var}(S) = p \cdot \left[ uS - E(S) \right]^2 + (1 - p) \cdot \left[ vS - E(S) \right]^2$$

其中 $E(S) = p \cdot u \cdot S + (1 - p) \cdot v \cdot S$。

最后令 $\text{Var}(dS) = \text{Var}(S)$（方差匹配）。

把 $u$、$v$、$p$ 看成三个未知数；把 $\mu$ 和 $\sigma$ 看成两个参数。这样三个未知数，两个方程，有无数个解。

在金融领域，为简便运算，取一组合适的解：令 $u + v = 2$（涨跌幅度对称），并要求 $uv \approx 1$（**一阶近似**：精确地 $uv = 1 - \sigma^2 \delta t$，与 1 只差二阶小量，可忽略）。于是取：

$$u = 1 + \sigma \sqrt{\delta t}, \quad v = 1 - \sigma \sqrt{\delta t}$$

代入上面第一个方程简化后得：

$$p = \frac{1}{2} + \frac{\mu}{2 \sigma}\sqrt{\delta t}$$

另外我们可以把 $u$、$v$ 直接带入之前的 $p'$ 公式，化简后居然得到了：

$$p' = \frac{1}{2} + \frac{r}{2 \sigma}\sqrt{\delta t}$$

也就是说，风险中性概率与真实概率的不同点在于收益率上：真实世界是实际收益率 $\mu$，风险中性世界是无风险收益率 $r$。

从这里我们也可以知道，**期权的价值就是在风险中性世界中未来价值期望的折现**。(2)

### 十五、多重二叉树

把单步二叉树重复 $N$ 期，就得到多重二叉树：

- 涨 $k$ 次、跌 $N-k$ 次的终端价格为 $S_0 u^{k} v^{N-k}$；由于 $v < 1$，上方路径价格越来越大，下方路径价格越来越小、趋近于 0。

- 关键性质：乘法可交换，$uvS = vuS$ **恒成立**，所以"先涨后跌"与"先跌后涨"必然到达同一节点——树是**可重组的**（recombining tree）。这正是"上下增长幅度取对称"的用意：节点数从 $2^N$ 条路径压缩为 $\frac{(N+1)(N+2)}{2}$ 个，指数级计算量降为平方级。

它有什么用？**欧式期权定价**。从期末开始，一步一步往回推：期末每个节点按 $V_T = \max\{S_T - K, 0\}$ 定值，再逐层用

$$V = \frac{1}{1 + r \cdot \delta t} \cdot \left[ p' \cdot V^{+} + (1 - p') \cdot V^{-} \right]$$

倒推回 $t = 0$，就是我们之前的方法。

我们是时候从离散扩展到连续了。

### 十六、从二叉树到连续：BS 方程

先回忆一下之前的定价模型：

$$V = \frac{1}{1 + r \cdot \delta t} \cdot \left[ p' \cdot V^{+} + (1 - p') \cdot V^{-} \right]$$

其中：

$$V^{+} = V(uS, \, t + \delta t) = V\left( (1 + \sigma \sqrt{\delta t}) \cdot S, \, t + \delta t \right)$$

泰勒展开得：

$$V^{+} = V(S, t) + \frac{\partial V}{\partial t} \delta t + \frac{\partial V}{\partial S} \sigma S \sqrt{\delta t} + \frac{1}{2} \frac{\partial^2 V}{\partial S^2} \sigma^2 S^2 \delta t + o(\delta t)$$

以及：

$$V^{-} = V(vS, \, t + \delta t) = V\left( (1 - \sigma \sqrt{\delta t}) \cdot S, \, t + \delta t \right)$$

泰勒展开得：

$$V^{-} = V(S, t) + \frac{\partial V}{\partial t} \delta t - \frac{\partial V}{\partial S} \sigma S \sqrt{\delta t} + \frac{1}{2} \frac{\partial^2 V}{\partial S^2} \sigma^2 S^2 \delta t + o(\delta t)$$

把 $p' = \frac{1}{2} + \frac{r}{2 \sigma}\sqrt{\delta t}$ 代入进去，忽略 $o(\delta t)$，且令 $\frac{1}{1 + r \cdot \delta t} \approx 1 - r \cdot \delta t$，这个 $V(S, t)$ 方程就会变成：

$$V(S,t) = (1 - r \delta t) \cdot \left[ V(S,t) + \frac{\partial V}{\partial t} \delta t + \frac{1}{2} \frac{\partial^2 V}{\partial S^2} \sigma^2 S^2 \delta t + \frac{r}{2 \sigma}\sqrt{\delta t} \cdot 2 \cdot \frac{\partial V}{\partial S} \sigma S \sqrt{\delta t} \right]$$

$$= V - rV \delta t + \frac{\partial V}{\partial t} \delta t + \frac{1}{2} \frac{\partial^2 V}{\partial S^2} \sigma^2 S^2 \delta t + r S \frac{\partial V}{\partial S} \delta t - \frac{1}{2} \frac{\partial^2 V}{\partial S^2} r \sigma^2 S^2 \delta t^2 - r^2 S \frac{\partial V}{\partial S} \delta t^2$$

忽略 $o(\delta t)$，再消掉 $\delta t$ 得：

$$\frac{\partial V}{\partial t} + \frac{1}{2} \sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} + r S \frac{\partial V}{\partial S} - r V = 0$$

**这句是大名鼎鼎的 BS 方程。**

其中各项含义：

- $\frac{\partial V}{\partial t}$：期权随时间的变化率
- $\frac{1}{2} \sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}$：波动率对期权价值的影响
- $r S \frac{\partial V}{\partial S}$：期权价值的无风险增长
- $r V$：期权价值的无风险折现

这样完整的方程的实际意义是：

> 期权的时间价值 + 波动率对期权价值的影响 + 期权价值的无风险增长 = 期权价值的无风险折现

注：其实这个推导在数学上不是完全严谨，后续也会有补充。